# 07 RMSNorm and Residual Connections

## Problem

为什么 Transformer 不是把 attention 和 FFN 一层层硬堆起来就行？Residual 和 RMSNorm 到底分别解决了什么问题，为什么现代 LLM 里经常看到 pre-norm？

## Dependencies

建议先理解 attention、多头注意力和基础张量 shape。


## Goals

- 理解 residual 为什么能帮助信息和梯度更顺畅地穿过深层网络
- 理解 LayerNorm 与 RMSNorm 的差异
- 理解 pre-norm 架构为什么在现代 LLM 中常见
- 能通过最小例子解释 norm 和 residual 为什么不是配角

## Scope and Approach

这一节的重点不是记住几个层名，而是回答一个非常实际的问题：为什么深层模型不加这些结构会更难训？我们会分别看 residual、LayerNorm、RMSNorm，再看它们如何在 block 里配合工作。


## 为什么深层网络会不稳定

如果把很多层神经网络简单串起来，每一层都在不断改写表示，问题就会逐渐出现：

- 前面的信息可能被后面的层越改越偏
- 数值规模可能越来越大或越来越小
- 梯度传播会越来越别扭

Residual 和 norm 的角色就是：

- residual：保留一条更直接的信息通路
- norm：把数值规模维持在更可控的范围内

没有它们，层堆深以后模型常常更难训稳。


## residual 到底在干什么

Residual 的形式很简单：

- 子层算出一个更新量
- 再把这个更新量加回原输入

也就是说，它不是让子层“从零重写一份表示”，而是让子层更像在说：

**“我在原表示基础上做一点修正。”**

这样做的好处是：

- 原始信息不会轻易丢掉
- 优化时更容易学成“小步修改”而不是“彻底重建”
- 梯度也更容易通过这条捷径往前传


In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)

# 一个非常小的 residual 例子。
hidden = np.array([0.2, 0.4, 0.6, 0.8])
update = np.array([-0.1, 0.3, 0.0, 0.2])

# 如果没有 residual，子层输出直接替代原表示。
without_residual = update

# 如果有 residual，子层输出被当作“修正量”加回原表示。
with_residual = hidden + update

print('hidden =', hidden)
print('update =', update)
print('without_residual =', without_residual)
print('with_residual    =', with_residual)


## norm 又在干什么

norm 关心的是数值尺度。因为一层层计算后，有些维度可能越来越大，有些越来越小，这会让后面层的输入分布变得难以控制。

LayerNorm 的经典做法是：

- 先减均值
- 再除标准差

RMSNorm 更简洁，它只关心均方根规模，不显式减均值。你可以先把它理解成：

- LayerNorm 在做“居中 + 缩放”
- RMSNorm 更像只做“缩放到合理范围”


In [ ]:
x = np.array([
    [2.0, 4.0, 6.0, 8.0],
    [1.0, 3.0, 5.0, 7.0],
], dtype=float)

def layer_norm(x, eps=1e-6):
    # LayerNorm 先减均值，再按标准差做归一化。
    mean = x.mean(axis=-1, keepdims=True)
    var = ((x - mean) ** 2).mean(axis=-1, keepdims=True)
    return (x - mean) / np.sqrt(var + eps)

def rms_norm(x, eps=1e-6):
    # RMSNorm 不显式减均值，只看均方根大小。
    rms = np.sqrt((x ** 2).mean(axis=-1, keepdims=True) + eps)
    return x / rms

print('LayerNorm =')
print(layer_norm(x))
print('RMSNorm =')
print(rms_norm(x))


## 为什么现代 LLM 里经常看到 RMSNorm

RMSNorm 在很多现代 LLM 里常见，不是因为 LayerNorm 错了，而是因为它更简洁，同时在很多场景里也足够稳定。它保留了“控制数值尺度”的关键作用，但形式更轻一些。

你可以先把它理解成一种工程取舍：

- 我最关心的是别让尺度乱飞
- 我不一定每次都要显式做减均值

这也是为什么它在现代 decoder-only 架构里很常见。


In [ ]:
# 一个最小 pre-norm 流程示意。
# pre-norm 的意思是：先做 norm，再进入子层，然后再 residual 加回去。
hidden = np.array([0.2, 0.4, 0.6, 0.8])
normed = rms_norm(hidden[None, :])[0]

# 这里用一个很简单的逐元素缩放来模拟“子层更新”。
sub_layer_output = normed * np.array([1.2, 0.8, 1.0, 1.1])
pre_norm_residual = hidden + sub_layer_output

print('hidden =', hidden)
print('normed =', normed)
print('sub_layer_output =', sub_layer_output)
print('pre_norm_residual =', pre_norm_residual)


## 为什么 pre-norm 常见

pre-norm 的数据流是：

- 先 norm
- 再进 attention 或 FFN 子层
- 再 residual 加回去

它在工程上常见的原因之一，是它往往更容易把深层训练稳定下来。至少在理解阶段，你可以先记成：

- pre-norm 先把输入数值整理一下
- 再让子层去处理一个更可控的输入分布


## Common mistakes

- 只记住“残差有用”，却说不清它为什么有用。关键在于它保留了一条更直接的信息路径。
- 把 LayerNorm 和 RMSNorm 当成完全等价。它们目标相近，但处理方式不同。
- 忽视 norm 放在子层前还是后。pre-norm 与 post-norm 在训练稳定性上会有差别。
- 以为 residual 和 norm 只是工程细节。实际上它们是模型能否堆深的重要前提。

## Checkpoints

- 把输入 `x` 改成均值不是 0 的数据，比较 LayerNorm 和 RMSNorm 输出差异。
- 用自己的话解释为什么深层网络如果没有 residual，优化会更困难。
- 自己画一张 pre-norm Transformer block 的数据流图。
- 回答：RMSNorm 主要是在改变信息内容，还是在控制数值尺度？

## Summary

RMSNorm 和 residual 不是配角，它们是现代 LLM 能稳定堆深的重要基础设施。Residual 保证信息和梯度有一条更直接的通路，norm 保证数值尺度更可控。没有这些结构，attention 再聪明，深层网络也很难稳定工作。

## Next Module

下一节进入 FFN 和 SwiGLU。attention 负责跨位置交换信息，而 FFN 负责每个位置内部的非线性加工。
